In [ ]:
# ======================================================================
# NOTEBOOK DE TREINAMENTO - PROJETO PASSOS MÁGICOS
# Objetivo: Processar dados de 2024 e gerar o binário do modelo (.pkl)
# ======================================================================

import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. CARREGAMENTO DOS DADOS (Caminho relativo para portabilidade)
# Considera que o notebook está em uma subpasta (ex: /notebooks)
caminho_dados = os.path.join('..', 'data', 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx')

# Caso o notebook esteja na raiz, use: caminho_dados = os.path.join('data', '...')
if not os.path.exists(caminho_dados):
    caminho_dados = os.path.join('data', 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx')

df_24 = pd.read_excel(caminho_dados, sheet_name='PEDE2024')

# 2. SELEÇÃO E TRATAMENTO DE FEATURES
# Criamos uma lista dos indicadores base que o modelo precisa
indicadores_base = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'INDE']
colunas_encontradas = {}

# Mapeamento dinâmico: procura a coluna que contém o nome do indicador
for ind in indicadores_base:
    col_encontrada = [c for c in df_24.columns if ind in str(c)]
    if col_encontrada:
        # Pega a primeira ocorrência encontrada (ex: 'IAN 2024' ou apenas 'IAN')
        colunas_encontradas[ind] = col_encontrada[0]
    else:
        print(f"⚠️ Alerta: Indicador {ind} não localizado nas colunas!")

# Filtra o DataFrame usando os nomes reais encontrados no Excel
df_ml = df_24[list(colunas_encontradas.values())].copy()

# Renomeia para o padrão simples (ex: 'IAN 2024' -> 'IAN') 
# Isso garante que o modelo aprenda com nomes limpos, iguais aos do Dashboard
df_ml.columns = [next(k for k, v in colunas_encontradas.items() if v == col) for col in df_ml.columns]

# Padronização numérica (converte vírgulas e remove vazios)
for col in df_ml.columns:
    df_ml[col] = pd.to_numeric(df_ml[col].astype(str).str.replace(',', '.'), errors='coerce')

df_ml = df_ml.dropna()

# 3. DEFINIÇÃO DO ALVO (TARGET)
# Como renomeamos as colunas no passo anterior, agora usamos apenas 'INDE'
# Criamos a classificação de risco: 1 para INDE abaixo de 6.0, 0 para os demais
df_ml['target'] = (df_ml['INDE'] < 6.0).astype(int)

# Definimos X (entrada) e y (saída)
X = df_ml[['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP']] # Apenas as colunas de notas
y = df_ml['target']

# 4. TREINAMENTO DO MODELO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Usamos class_weight='balanced' pois geralmente temos menos alunos em risco que alunos estáveis
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
modelo_rf.fit(X_train, y_train)

# 5. VALIDAÇÃO SIMPLES
previsoes = modelo_rf.predict(X_test)
print(f"✅ Acurácia do Modelo: {accuracy_score(y_test, previsoes)*100:.2f}%")
print("\nRelatório de Classificação:\n", classification_report(y_test, previsoes))

# 6. EXPORTAÇÃO DO MODELO (.PKL)
# Salva na pasta /models para que o app.py possa consumir
caminho_salvar = os.path.join('..', 'models', 'modelo_passos_magicos.pkl')

# Garante a existência da pasta
if not os.path.exists(os.path.dirname(caminho_salvar)):
    os.makedirs(os.path.dirname(caminho_salvar))

caminho_salvar = os.path.join('..', 'models', 'modelo_passos_magicos.pkl')
joblib.dump(modelo_rf, caminho_salvar)

✅ Acurácia do Modelo: 97.16%

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98       196
           1       0.85      0.73      0.79        15

    accuracy                           0.97       211
   macro avg       0.91      0.86      0.89       211
weighted avg       0.97      0.97      0.97       211


🚀 Sucesso! Modelo exportado para: ..\models\modelo_passos_magicos.pkl
